In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\User\Desktop\CreditRiskLearning\Exploration
['exploration.ipynb', 'modelling.ipynb']


In [3]:
# Build path relative to notebook location
base_path = os.path.dirname(os.path.abspath('exploration.ipynb'))
data_path = os.path.join(base_path, 'German Credit Data', 'german.data')


In [4]:
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status', 'other_debtors',
    'residence_since', 'property', 'age', 'other_installments', 'housing',
    'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker', 'target'
]



In [5]:
# Load data
df = pd.read_csv('../German Credit Data/german.data', sep=' ', header=None, names=columns)

# Map target FIRST before anything else
df['target'] = df['target'].map({1: 0, 2: 1})

# Verify it worked
print(df['target'].unique())  # should show [0 1]

# Then encode
cat_columns = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)

# Then split
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify split target values
print(y_test.unique())  # should show [0 1]

[0 1]


C:\Users\User\AppData\Local\Temp\ipykernel_37096\195248330.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(include='object').columns.tolist()


NameError: name 'train_test_split' is not defined

In [ ]:
df.dtypes

In [ ]:
# Get list of categorical items
cat_columns = df.select_dtypes(include='object').columns.tolist()
print(cat_columns)

# One hoit encode items
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)

print(f'Original shape: {df.shape}')
print(f'Encoded shape: {df_encoded.shape}')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Split into features and target
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']


# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training rows: {X_train.shape[0]}')
print(f'Testing rows: {X_test.shape[0]}')

In [ ]:
# Build and train the model

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Make predictions

y_pred = lr_model.predict(X_test)

print('Model trained successfully!')


In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Retrain with scaled data and more iterations

lr_model = LogisticRegression(max_iter = 2000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)

print('Model trained successfully!')

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# Get probabilities instead of hard predictions

y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# Try a lower threshold

threshold = 0.4
y_pred_tuned = (y_prob >= threshold).astype(int)

print(f'Results at threshold {threshold}:')
print(classification_report(y_test, y_pred_tuned))
